In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import seaborn as sns

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import configure_mpl

configure_mpl(Path("../fonts/"))

RANDOM_SEED = 202607101941

DATA_PATH = Path("../reports/thesis/results/data/model/all_interventions/")
schema = schema.post_index()

rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
schema.get_short_names("measurement")

In [ ]:
null_measurements_asym = np.load(DATA_PATH / "ising_00.npz")["measurements"][
    ..., 5, :, 7
]
measurements_asym = np.load(DATA_PATH / "ising_80.npz")["measurements"][..., 5, :, 7]

null_measurements_sym = np.load(DATA_PATH / "sym_ising_00.npz")["measurements"][
    ..., 5, :, 7
]
measurements_sym = np.load(DATA_PATH / "sym_ising_80.npz")["measurements"][..., 5, :, 7]

In [ ]:
collective_effect_asym = (measurements_asym - null_measurements_asym).mean(axis=1)
collective_effect_asym = (measurements_asym - null_measurements_asym).mean(axis=1)

collective_effect_sym = (measurements_sym - null_measurements_sym).mean(axis=1)
collective_effect_sym = (measurements_sym - null_measurements_sym).mean(axis=1)

Compute ECDF

In [ ]:
@dataclass
class ECDF:
    values: npt.NDArray[np.float64]
    probs: npt.NDArray[np.float64]

    @classmethod
    def from_arr(cls, arr: npt.NDArray[np.float64]) -> ECDF:
        assert arr.ndim == 1, "Expected 1D array"
        values, counts = np.unique_counts(arr)
        cumsum = counts.cumsum()
        probs = cumsum / counts.sum()
        return cls(values, probs)

    def sample(self, n: int, rng: np.random.Generator) -> npt.NDArray[np.float64]:
        u = rng.random(n)
        idxes = np.argmax(self.probs[:, None] >= u, axis=0)
        return self.values[idxes]

In [ ]:
def estimate_prob_lt_independent(
    arr1: npt.NDArray[np.float64],
    arr2: npt.NDArray[np.float64],
    rng: np.random.Generator,
    n: int = 10_000_000,
) -> np.float64:
    samples_1 = ECDF.from_arr(arr1).sample(n, rng)
    samples_2 = ECDF.from_arr(arr2).sample(n, rng)
    return (samples_1 < samples_2).sum() / n

In [ ]:
def bootstrap_lt_empirical_dist(
    arr1: npt.NDArray[np.float64],
    arr2: npt.NDArray[np.float64],
    rng: np.random.Generator,
    n: int = 100_000,
) -> npt.NDArray[np.float64]:
    assert arr1.shape == arr2.shape
    rows = arr1.shape[0]
    idxes = rng.choice(np.arange(rows), (n, rows), replace=True)
    bootstraps_arr1 = arr1[idxes]
    bootstraps_arr2 = arr2[idxes]
    estimates = (bootstraps_arr1 < bootstraps_arr2).sum(axis=1) / rows
    return estimates

### Symmetric model

Politics and CC Real

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 2), constrained_layout=True)

independent_prob_est = estimate_prob_lt_independent(
    collective_effect_sym[:, 5], collective_effect_sym[:, 0], rng
)
dependent_prob_ests = bootstrap_lt_empirical_dist(
    collective_effect_sym[:, 5], collective_effect_sym[:, 0], rng
)

ci_interval = np.percentile(dependent_prob_ests, (2.5, 97.5))
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.6,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=ci_interval,
)
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.3,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=(0, ci_interval[0]),
)
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.3,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=(ci_interval[1], None),
)

ax.set_xlim(0, None)
ax.axvline(x=independent_prob_est)

ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)

ax.set_xlabel("Probability 'Politics' < 'CC Real' (symm)")

CC Real and CC Human

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 2), constrained_layout=True)

independent_prob_est = estimate_prob_lt_independent(
    collective_effect_sym[:, 0], collective_effect_sym[:, 1], rng
)
dependent_prob_ests = bootstrap_lt_empirical_dist(
    collective_effect_sym[:, 0], collective_effect_sym[:, 1], rng
)

ci_interval = np.percentile(dependent_prob_ests, (2.5, 97.5))
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.6,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=ci_interval,
)
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.3,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=(0, ci_interval[0]),
)
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.3,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=(ci_interval[1], None),
)

ax.set_xlim(0, None)
ax.axvline(x=independent_prob_est)

ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)

ax.set_xlabel("Probability 'CC Real' < 'CC Human' (symm)")

### Asymmetric model

CC Real and CC Human

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 2), constrained_layout=True)

independent_prob_est = estimate_prob_lt_independent(
    collective_effect_asym[:, 0], collective_effect_asym[:, 1], rng
)
dependent_prob_ests = bootstrap_lt_empirical_dist(
    collective_effect_asym[:, 0], collective_effect_asym[:, 1], rng
)

ci_interval = np.percentile(dependent_prob_ests, (2.5, 97.5))
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.6,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=ci_interval,
)
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.3,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=(0, ci_interval[0]),
)
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.3,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=(ci_interval[1], None),
)

ax.set_xlim(0, None)
ax.axvline(x=independent_prob_est)

ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)

ax.set_xlabel("Probability 'CC Real' < 'CC Human' (asym)")

CC Human and Weather Worry

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 2), constrained_layout=True)

independent_prob_est = estimate_prob_lt_independent(
    collective_effect_asym[:, 1], collective_effect_asym[:, 4], rng
)
dependent_prob_ests = bootstrap_lt_empirical_dist(
    collective_effect_asym[:, 1], collective_effect_asym[:, 4], rng
)

ci_interval = np.percentile(dependent_prob_ests, (2.5, 97.5))
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.6,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=ci_interval,
)
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.3,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=(0, ci_interval[0]),
)
sns.kdeplot(
    dependent_prob_ests,
    fill=True,
    alpha=0.3,
    bw_adjust=3.0,
    color="tab:blue",
    ax=ax,
    clip=(ci_interval[1], None),
)

ax.set_xlim(0, None)
ax.axvline(x=independent_prob_est)

ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)

ax.set_xlabel("Probability 'CC Human' < 'Weather Worry' (asym)")